# Qiskit Hello World — on IBM Quantum hardware

Builds a Bell-pair circuit, then estimates the $\langle ZZ \rangle$ observable on it using the IBM Quantum Runtime Estimator.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.executor_estimator import Estimator

## 1. Load credentials from `.env`
Credentials come from `.env` (`IBM_QUANTUM_TOKEN`), and optionally `IBM_QUANTUM_BACKEND`. Do NOT hardcode your token in source code or commit your `.env` file.

In [ ]:
# Resolved relative to this notebook so it works from any working
# directory, not just the project root.
load_dotenv(Path.cwd() / ".env")
token = os.environ.get("IBM_QUANTUM_TOKEN")
pinned = os.environ.get("IBM_QUANTUM_BACKEND") or None

if not token:
    raise ValueError(
        "IBM_QUANTUM_TOKEN not found. Create a .env in the project root:\n"
        "  IBM_QUANTUM_TOKEN=<your 44-character key>\n"
        "  IBM_QUANTUM_BACKEND=ibm_kingston   # optional"
    )

## 2. Save the account (one-time)

`QiskitRuntimeService.save_account()` writes the credentials to disk so later sessions authenticate automatically.
Once you run it, you don't need to call this cell every time.

In [ ]:
QiskitRuntimeService.save_account(
    # For `token`, use the 44-character API_KEY you created
    # and saved from the IBM Quantum Platform Home dashboard
    token=token,
    overwrite=True,
)

## 3. Authenticate

Run this every time you need the service. With credentials already saved,
`QiskitRuntimeService()` picks them up with no arguments.

In [ ]:
service = QiskitRuntimeService()
print("Connected to:", service.channel)

### Available backends

Backends the account can reach.

In [ ]:
for _backend in service.backends():
    try:
        _status = _backend.status()
        _pending = getattr(_status, "pending_jobs", "n/a")
        _msg = getattr(_status, "status_msg", "")
    except Exception:
        _pending, _msg = "n/a", ""
    print(f"{_backend.name:24} pending_jobs={_pending} {_msg}")

### Pick a backend

Set `IBM_QUANTUM_BACKEND` in `.env` to pin a specific device (e.g. `ibm_kingston`).
Leave it unset to let `least_busy` choose the open-plan device with the
shortest queue. Falls back to a local fake backend if neither works.

In [ ]:
backend = None

if pinned:
    try:
        backend = service.backend(pinned)
        print(f"Pinned backend: {backend.name}")
    except Exception as exc:
        print(f"Could not load '{pinned}' ({type(exc).__name__}); falling back.")

if backend is None:
    try:
        backend = service.least_busy(operational=True, simulator=False)
        print("Auto-selected backend:", backend.name)
    except Exception as exc:  # no open-plan access
        print(f"No real backend available ({type(exc).__name__}); using a local fake.")

if backend is None:
    from qiskit_ibm_runtime.fake_provider import FakeBelemV2

    backend = FakeBelemV2()
    print("Fake backend:", backend.name)

# Fake backends have no .status(); real ones do.
_st = getattr(backend, "status", None)
if _st is not None:
    print("Num qubits:", backend.num_qubits, "| status:", getattr(_st(), "status_msg", "?"))
else:
    print("Num qubits:", backend.num_qubits)

## 4. Build the circuit

Simple Bell pair: A Hadamard on qubit 0 then a CNOT onto qubit 1

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

In [ ]:
qc.draw("mpl")

In [ ]:
# Ideal state vector, computed locally — no hardware needed.
# Basis order is |00>, |01>, |10>, |11>.
sv = Statevector.from_instruction(qc)
print(f"Statevector: {sv.data}")
print(f"Probabilities: {dict(sv.probabilities_dict())}")

### Observable

$\langle Z \otimes Z \rangle$ on the Bell pair. Expected value is $+1$ for $\lvert \Phi^+ \rangle$ — the qubits are perfectly correlated.

In [ ]:
observable = SparsePauliOp("ZZ")

# The exact noiseless answer, for comparison against the hardware result.
print(f"Ideal <ZZ> (exact): {sv.expectation_value(observable).real}")

## 5. Transpile to the backend

Maps the circuit and the observable onto the backend's physical qubits.

In [ ]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
isa_observable = observable.apply_layout(isa_circuit.layout)

isa_circuit.draw("mpl")

## 6. Run the estimator

Submits to the backend and waits for the result. This consumes real quantum-compute time on your account.

In [ ]:
estimator = Estimator(mode=backend)
job = estimator.run([(isa_circuit, isa_observable)])
result = job.result()

In [ ]:
print(f"Expectation value: {result[0].data.evs}")

### Expectation value

Should be $\approx +1$ — confirming the Bell pair is in the $\lvert \Phi^+ \rangle$ state with perfect $Z$-parity correlation. A two-qubit $ZZ$ expectation value is always bounded by $[-1, +1]$; anything outside that range means the estimator is being read incorrectly.

Real hardware returns slightly less than the ideal `1.0` printed above, due to gate error, measurement error, and finite sampling.

In [ ]:
result[0].data.evs